## 1. Load raw data

In [16]:
import re
import numpy as np
import pandas as pd

city_df = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\bronze\city_df.csv')
province_df = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\bronze\province_df.csv')
housing_raw = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\bronze\foreclosed_housing.csv')

dim_geo = pd.read_csv(
    r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\silver\dim_geo.csv',
    low_memory=False,
    dtype={'cityCode': 'Int64', 'provinceCode': 'Int64'}  # Int64 (capital I) allows NaN in an int column
)
dim_geo.count()


cityCode        146
isCapital       146
provinceCode    146
cityName        146
provinceName    146
islandGroup     146
dtype: int64

In [21]:
  # should be back to ~150
print(len(dim_geo))
print(dim_geo.tail())

dim_geo = dim_geo.dropna(how='all')          # drop fully-blank rows
dim_geo = dim_geo[dim_geo['cityCode'].notna()]  # belt-and-suspenders: real rows always have a cityCode

print(len(dim_geo))
dim_geo.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\silver\dim_geo.csv',index=False)


146
      cityCode isCapital  provinceCode  cityName     provinceName islandGroup
141  166803000     False     166800000    bislig  surigao del sur    mindanao
142  166819000      True     166800000    tandag  surigao del sur    mindanao
143  150702000      True     150700000   lamitan          basilan    mindanao
144  153617000      True     153600000    marawi    lanao del sur    mindanao
145  129804000     False     124700000  cotabato   north cotabato    mindanao
146


## 2. Filter housing (Bronze -> working Silver base)

In [2]:
housing_filtered = housing_raw[
    (housing_raw['propertyType'] == 'residential') &
    (housing_raw['price'] >= 10000)
][['id', 'sourceSlug', 'sourceName', 'title', 'price', 'priceFormatted',
    'pricePerSqm', 'floorArea', 'lotArea', 'city', 'province',
    'isNew', 'daysListed', 'listingScore', 'firstSeenAt']].copy()

print(f'Number of residential property: {len(housing_filtered)}')
housing_filtered.head()


Number of residential property: 20274


,id,sourceSlug,sourceName,title,price,priceFormatted,pricePerSqm,floorArea,lotArea,city,province,isNew,daysListed,listingScore,firstSeenAt
0,b3d16489-85e2-4652-9458-469f0c5d700d,metrobank,Metrobank,Townhouse,5769000.0,₱ 5.8M,52445.0,165.00,110.0,Mandaue,Cebu,False,63,140.0,2026-05-02T12:56:29.709Z
1,25fa12f6-bcf2-4a0a-9c47-cdd41ea94598,metrobank,Metrobank,With Improvement,5427000.0,₱ 5.4M,414.0,1117.00,13111.0,Bay,Laguna,False,63,135.0,2026-05-02T13:00:32.178Z
2,34d3d3c6-c96c-4ea9-ad7c-8c11d0037370,metrobank,Metrobank,With Improvement,7188000.0,₱ 7.2M,59900.0,296.00,120.0,Lapu-Lapu,Cebu,False,63,135.0,2026-05-02T12:54:01.353Z
5,5e85bcbc-70ae-4747-9287-72d499ae4ebd,metrobank,Metrobank,Condominium,27705000.0,₱ 27.7M,145235.0,190.76,NaN,Manila,Metro Manila,False,63,131.4,2026-05-02T13:04:52.502Z
6,1876579a-7fee-462e-a3ce-afda1e8501c4,metrobank,Metrobank,With Improvement,6601000.0,₱ 6.6M,6067.0,429.00,1088.0,Plaridel,Bulacan,False,63,130.0,2026-05-02T13:00:37.411Z


## 3. Build city/province name lists (for title-regex extraction)

In [3]:
# Build city_list and province_list from PSGC bronze, stripping the "City of" style prefixes
city_list = (
    city_df['name'].dropna().str.strip()
    .apply(lambda x: re.sub(r'^(City of|Science City of|Island Garden City of)\s+', '', x, flags=re.IGNORECASE).strip())
    .tolist()
)

province_list = province_df['name'].dropna()
province_list = province_list[province_list != ''].tolist()

print("Number of cities:", len(city_list))
print("Number of provinces:", len(province_list))


Number of cities: 146
Number of provinces: 81


## 4. Extract city/province from title where missing

In [4]:
city_pattern = '|'.join(re.escape(c) for c in sorted(city_list, key=len, reverse=True))
province_pattern = '|'.join(re.escape(p) for p in sorted(province_list, key=len, reverse=True))

# City: only extract where city is currently missing, check every such row
housing_filtered['cityFromTitle'] = np.where(
    housing_filtered['city'].isna() | (housing_filtered['city'] == ''),
    housing_filtered['title'].str.extract(f'(?i)({city_pattern})', expand=False),
    None
)
housing_filtered['cityFromTitle'] = housing_filtered['cityFromTitle'].replace('', np.nan)
housing_filtered['city'] = housing_filtered['city'].replace('', np.nan).fillna(housing_filtered['cityFromTitle'])

# Province: same independent treatment
housing_filtered['provinceFromTitle'] = np.where(
    housing_filtered['province'].isna() | (housing_filtered['province'] == ''),
    housing_filtered['title'].str.extract(f'(?i)({province_pattern})', expand=False),
    None
)
housing_filtered['provinceFromTitle'] = housing_filtered['provinceFromTitle'].replace('', np.nan)
housing_filtered['province'] = housing_filtered['province'].replace('', np.nan).fillna(housing_filtered['provinceFromTitle'])

housing_silver = housing_filtered[
    housing_filtered['city'].notna() | housing_filtered['province'].notna()
].copy()

print("Rows with city or province after enrichment:", len(housing_silver))


Rows with city or province after enrichment: 17911


## 5. Normalize province text (NCR -> Metro Manila, drop bad values, lowercase)

In [5]:
housing_silver['province'] = housing_silver['province'].apply(
    lambda x: 'Metro Manila' if isinstance(x, str) and re.match(r'(?i)^Ncr', x) else x
)
housing_silver.loc[housing_silver['province'] == 'Mindoro', 'province'] = np.nan

housing_silver['province'] = housing_silver['province'].str.lower()
housing_silver['city'] = housing_silver['city'].str.lower()
housing_silver = housing_silver.drop(columns=['provinceFromTitle'])


## 6. Clean city text (parentheses, "City of" prefix, trailing "City", sta./sto.)

In [6]:
housing_silver['cityClean'] = housing_silver['city'].str.replace(r'\s*\([^)]*\)', '', regex=True)
housing_silver['cityClean'] = housing_silver['cityClean'].str.replace(r'[,\.]+$', '', regex=True).str.strip()
housing_silver['cityClean'] = housing_silver['cityClean'].str.replace(r'(?i)^(city of|municipality of)\s+', '', regex=True)
housing_silver['cityClean'] = housing_silver['cityClean'].str.replace(r'(?i)\s+city$', '', regex=True)
housing_silver['cityClean'] = housing_silver['cityClean'].str.replace(r'(?i)\s+i/ii$', '', regex=True).str.strip()

housing_silver['cityClean'] = housing_silver['cityClean'].str.replace(r'(?i)^sta\.', 'santa', regex=True)
housing_silver['cityClean'] = housing_silver['cityClean'].str.replace(r'(?i)^sto\.', 'santo', regex=True)

housing_silver['city'] = housing_silver['cityClean']
housing_silver = housing_silver.drop(columns=['cityClean', 'cityFromTitle'])

housing_silver[['id', 'city', 'province']].head(10)


,id,city,province
0,b3d16489-85e2-4652-9458-469f0c5d700d,mandaue,cebu
1,25fa12f6-bcf2-4a0a-9c47-cdd41ea94598,bay,laguna
2,34d3d3c6-c96c-4ea9-ad7c-8c11d0037370,lapu-lapu,cebu
5,5e85bcbc-70ae-4747-9287-72d499ae4ebd,manila,metro manila
6,1876579a-7fee-462e-a3ce-afda1e8501c4,plaridel,bulacan
7,81ff202d-3797-4f3d-ae17-bb38a6587d14,manila,metro manila
8,83da7f76-afaa-4348-8ec3-13f3b13bdc23,general trias,cavite
9,6101a83c-e2c5-4dd4-af3e-64e5e1ab83e7,tuguegarao,cagayan
10,6d310957-6a64-47c9-ba10-2097a2ae6bbb,cabanatuan,nueva ecija
11,5f26b16e-962c-43a9-9ece-f5fbb7538541,cabanatuan,nueva ecija


## 7. Prep dim_geo + lookup tables for FK matching

In [7]:
dim_geo['cityName'] = dim_geo['cityName'].str.lower().str.strip()
dim_geo['provinceName'] = dim_geo['provinceName'].str.lower().str.strip()

city_lookup = dim_geo[['cityCode', 'cityName', 'provinceCode', 'provinceName']].dropna(subset=['cityName'])
dupe_cities = set(city_lookup[city_lookup.duplicated('cityName', keep=False)]['cityName'])
province_lookup = dim_geo[['provinceCode', 'provinceName']].dropna(subset=['provinceName']).drop_duplicates('provinceName')


## 8. Two-stage geography FK matching

- **Stage 1a** — match `(city, province)` together (avoids the duplicate-city-name fan-out: naga/san carlos/san fernando/talisay).
- **Stage 1b** — city-only fallback for non-ambiguous names still unmatched.
- **Stage 2** — for rows where city never resolved (e.g. small municipalities not in `dim_geo`), search province directly.

In [8]:
# === STAGE 1a: match city, pulling provinceCode from that SAME matched row ===
housing_fk = housing_silver.merge(
    city_lookup.rename(columns={'cityCode': 'geographyFk_city', 'provinceCode': 'geographyFk_province'}),
    left_on=['city', 'province'], right_on=['cityName', 'provinceName'],
    how='left'
).drop(columns=['cityName', 'provinceName'])

# === STAGE 1b: city-only fallback for non-ambiguous names still unmatched ===
fallback_eligible = housing_fk['geographyFk_city'].isna() & ~housing_fk['city'].isin(dupe_cities)

city_only = city_lookup.drop_duplicates('cityName').rename(
    columns={'cityCode': 'geographyFk_city', 'provinceCode': 'geographyFk_province'}
)

fallback = housing_fk.loc[fallback_eligible, ['city']].merge(
    city_only[['cityName', 'geographyFk_city', 'geographyFk_province']],
    left_on='city', right_on='cityName', how='left'
)
fallback.index = housing_fk.loc[fallback_eligible].index

housing_fk.loc[fallback_eligible, 'geographyFk_city'] = fallback['geographyFk_city']
housing_fk.loc[fallback_eligible, 'geographyFk_province'] = fallback['geographyFk_province']

housing_fk['is_ambiguous_city'] = housing_fk['geographyFk_city'].isna() & housing_fk['city'].isin(dupe_cities)

# === STAGE 2: rows where city STILL didn't resolve — search province directly ===
still_no_city = housing_fk['geographyFk_city'].isna()

prov_match = housing_fk.loc[still_no_city, ['province']].merge(
    province_lookup, left_on='province', right_on='provinceName', how='left'
)
prov_match.index = housing_fk.loc[still_no_city].index

housing_fk.loc[still_no_city, 'geographyFk_province'] = (
    housing_fk.loc[still_no_city, 'geographyFk_province']
    .fillna(prov_match['provinceCode'])
)


## 9. Sanity checks

In [9]:
print('rows in:', len(housing_silver), '| rows out:', len(housing_fk))
print('city resolved:', housing_fk['geographyFk_city'].notna().sum())
print('province resolved:', housing_fk['geographyFk_province'].notna().sum())
print('ambiguous city (needs review):', housing_fk['is_ambiguous_city'].sum())
print('fully unresolved:', (housing_fk['geographyFk_city'].isna() & housing_fk['geographyFk_province'].isna()).sum())


rows in: 17911 | rows out: 17911
city resolved: 11737
province resolved: 17870
ambiguous city (needs review): 1
fully unresolved: 41


## 10. Save Silver output

In [10]:
housing_fk.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\silver\dim_housing.csv', index=False)
print(f"Saved {len(housing_fk)} rows to dim_housing.csv")


Saved 17911 rows to dim_housing.csv


## 11. Build Gold `housing_fact`

Selects the fact-grain columns explicitly (rather than drop-by-name) so it doesn't silently keep stray columns like `province` or `is_ambiguous_city`, and carries both geography FKs instead of the old single `geographyFk`.

In [11]:
housing_fact = housing_fk[
    ['id', 'geographyFk_city', 'geographyFk_province', 'title',
     'price', 'pricePerSqm', 'floorArea', 'lotArea']
].copy()

housing_fact.head()


,id,geographyFk_city,geographyFk_province,title,price,pricePerSqm,floorArea,lotArea
0,b3d16489-85e2-4652-9458-469f0c5d700d,72230000,72200000,Townhouse,5769000.0,52445.0,165.00,110.0
1,25fa12f6-bcf2-4a0a-9c47-cdd41ea94598,<NA>,43400000,With Improvement,5427000.0,414.0,1117.00,13111.0
2,34d3d3c6-c96c-4ea9-ad7c-8c11d0037370,72226000,72200000,With Improvement,7188000.0,59900.0,296.00,120.0
3,5e85bcbc-70ae-4747-9287-72d499ae4ebd,133900000,130000000,Condominium,27705000.0,145235.0,190.76,NaN
4,1876579a-7fee-462e-a3ce-afda1e8501c4,<NA>,31400000,With Improvement,6601000.0,6067.0,429.00,1088.0


## 12. Property category extraction

In [12]:
category_map = [
    ("With Improvement-Bundle", "Lot with Improvement"),
    ("With Improvement", "Lot with Improvement"),
    ("House & Lot", "House & Lot"),
    ("Town Villa", "House & Lot"),
    ("H&L", "House & Lot"),
    ("Townhouse", "Townhouse"),
    ("Town House", "Townhouse"),
    ("Town & Country", "House & Lot"),
    ("Condominium", "Condominium"),
    ("Subdivision", "House & Lot"),
    ("Vacant Lot", "Vacant Lot"),
    ("Village", "House & Lot"),
    ("Camella", "House & Lot"),
    ("Tower", "Condominium"),
    ("Villa", "House & Lot"),
    ("Residential Lot", "Lot with Improvement"),
    ("Residential", "House & Lot"),
    ("Woodlands", "House & Lot"),
    ("Duplex", "Duplex")
]

def extract_category(title):
    if pd.isna(title):
        return 'N/A'
    for keyword, category in category_map:
        if re.search(re.escape(keyword), title, flags=re.IGNORECASE):
            return category
    return 'N/A'

housing_fact['propertyCategory'] = housing_fact['title'].apply(extract_category)

print(housing_fact['propertyCategory'].value_counts(dropna=False))


propertyCategory
Townhouse               3983
N/A                     3846
House & Lot             3301
Condominium             3058
Vacant Lot              2641
Lot with Improvement     651
Duplex                   431
Name: count, dtype: int64


## 13. Save Gold output

In [13]:
# housing_fact.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\fact_housing.csv', index=False)
housing_fact.head()


,id,geographyFk_city,geographyFk_province,title,price,pricePerSqm,floorArea,lotArea,propertyCategory
0,b3d16489-85e2-4652-9458-469f0c5d700d,72230000,72200000,Townhouse,5769000.0,52445.0,165.00,110.0,Townhouse
1,25fa12f6-bcf2-4a0a-9c47-cdd41ea94598,<NA>,43400000,With Improvement,5427000.0,414.0,1117.00,13111.0,Lot with Improvement
2,34d3d3c6-c96c-4ea9-ad7c-8c11d0037370,72226000,72200000,With Improvement,7188000.0,59900.0,296.00,120.0,Lot with Improvement
3,5e85bcbc-70ae-4747-9287-72d499ae4ebd,133900000,130000000,Condominium,27705000.0,145235.0,190.76,NaN,Condominium
4,1876579a-7fee-462e-a3ce-afda1e8501c4,<NA>,31400000,With Improvement,6601000.0,6067.0,429.00,1088.0,Lot with Improvement


In [15]:
housing_fact.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\fact_housing.csv')